# Древние автомобили — Graph Attention для предсказания траектории

Полный воспроизводимый pipeline финального решения: CSV → локальная векторная карта → spatial validation → Small/Big Graph Attention → OOF-ансамбль → обучение на всех данных → `submission.csv`.

Итоговая честная spatial-метрика ансамбля: **2.8857 OOF ADE**. Подробное описание решений и экспериментов находится в `README.md`.

## 1. Конфигурация и зависимости

Notebook рассчитан на CUDA GPU с 24–32 GB VRAM. Все пути задаются относительно каталога notebook. При повторном запуске извлечённые NPZ-кэши используются повторно.

In [ ]:
from __future__ import annotations

import copy
import csv
import math
import random
import sys
import time
from pathlib import Path

import numpy as np
import torch
from scipy.optimize import minimize
from scipy.spatial import cKDTree
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

ROOT = Path.cwd()
DATA_DIR = ROOT / 'data'
ARTIFACT_DIR = ROOT / 'artifacts'
DATA_DIR.mkdir(exist_ok=True)
ARTIFACT_DIR.mkdir(exist_ok=True)

TRAIN_CSV = DATA_DIR / 'train.csv'
TEST_CSV = DATA_DIR / 'test.csv'
TRAIN_CACHE = DATA_DIR / 'train_tokens_graph.npz'
TEST_CACHE = DATA_DIR / 'test_tokens_graph.npz'

RUN_SPATIAL_VALIDATION = True
RUN_FULL_TRAINING = True
REUSE_EXISTING_ARTIFACTS = True

assert torch.cuda.is_available(), 'Для обучения финальной модели требуется CUDA GPU'
DEVICE = torch.device('cuda')
torch.set_float32_matmul_precision('high')
print('device:', torch.cuda.get_device_name(0))

def seed_all(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


## 2. Чтение CSV и построение 512 map-токенов

CSV содержит многострочные массивы произвольной длины. Для каждой сцены строится локальная система координат, выбираются четыре непересекающихся блока по 128 векторов и вычисляется граф из восьми ближайших соседей.

In [ ]:
BLOCK = 128
TOKEN_COUNT = 4 * BLOCK
NEIGHBOR_COUNT = 8
TRAJECTORY_STEPS = 46

def parse_array(text: str) -> np.ndarray:
    return np.fromstring(text.strip().strip('[]'), sep=' ', dtype=np.float32)

def padded(vectors: np.ndarray, indices: np.ndarray) -> np.ndarray:
    result = np.zeros((BLOCK, 5), dtype=np.float32)
    take = min(BLOCK, len(indices))
    if take:
        result[:take] = vectors[indices[:take]]
    return result

def take_unique(candidates: np.ndarray, used: set[int]) -> np.ndarray:
    selected = []
    for raw_index in candidates:
        index = int(raw_index)
        if index in used:
            continue
        used.add(index)
        selected.append(index)
        if len(selected) == BLOCK:
            break
    return np.asarray(selected, dtype=np.int64)

def build_neighbor_graph(tokens: np.ndarray) -> np.ndarray:
    valid = np.flatnonzero(np.abs(tokens).sum(axis=1) > 0)
    neighbors = np.tile(np.arange(len(tokens), dtype=np.uint16)[:, None], (1, NEIGHBOR_COUNT))
    if len(valid) <= 1:
        return neighbors
    tree = cKDTree(tokens[valid, :2])
    k = min(NEIGHBOR_COUNT + 1, len(valid))
    _, local_indices = tree.query(tokens[valid, :2], k=k)
    if k == 1:
        local_indices = local_indices[:, None]
    selected = valid[local_indices[:, 1:]]
    for column in range(selected.shape[1]):
        neighbors[valid, column] = selected[:, column]
    return neighbors

def extract_dataset(csv_path: Path, output_path: Path) -> None:
    csv.field_size_limit(sys.maxsize)
    tokens_all, contexts, frames, targets, graphs = [], [], [], [], []
    started = time.time()
    with csv_path.open('r', encoding='utf-8', newline='') as stream:
        reader = csv.reader(stream)
        header = next(reader)
        is_train = 'target_x' in header or len(header) >= 8
        for row_index, row in enumerate(reader):
            x, y, dx, dy, types = (parse_array(row[i]) for i in range(5))
            goal_global = parse_array(row[5])
            n = len(x)
            if not (len(y) == len(dx) == len(dy) == len(types) == n):
                raise ValueError(f'Несовпадающие длины массивов в строке {row_index}')

            x0, y0 = float(x[0]), float(y[0])
            norms = np.hypot(dx, dy)
            valid_direction = np.flatnonzero(norms >= 0.5)
            direction_index = int(valid_direction[0]) if len(valid_direction) else 0
            ux, uy = float(dx[direction_index]), float(dy[direction_index])
            norm = max(math.hypot(ux, uy), 1e-6)
            ux, uy = ux / norm, uy / norm

            local_x = (x - x0) * ux + (y - y0) * uy
            local_y = -(x - x0) * uy + (y - y0) * ux
            local_dx = dx * ux + dy * uy
            local_dy = -dx * uy + dy * ux
            vectors = np.stack((local_x / 100, local_y / 100, local_dx, local_dy, types - 15), axis=1)

            start_distance = local_x**2 + local_y**2
            start_candidates = np.argsort(start_distance)
            has_goal = len(goal_global) >= 2 and np.isfinite(goal_global[:2]).all()
            if has_goal:
                goal_offset_x = float(goal_global[0]) - x0
                goal_offset_y = float(goal_global[1]) - y0
                goal_along = goal_offset_x * ux + goal_offset_y * uy
                goal_lateral = -goal_offset_x * uy + goal_offset_y * ux
                goal_distance = (local_x - goal_along)**2 + (local_y - goal_lateral)**2
                goal_candidates = np.argsort(goal_distance)
                denominator = max(goal_along**2 + goal_lateral**2, 1e-6)
                projection = np.clip((local_x * goal_along + local_y * goal_lateral) / denominator, 0, 1)
                closest_x, closest_y = projection * goal_along, projection * goal_lateral
                corridor_distance = (local_x - closest_x)**2 + (local_y - closest_y)**2
                corridor_pool = np.argsort(corridor_distance)[:min(BLOCK * 4, n)]
                ordered = corridor_pool[np.argsort(projection[corridor_pool])]
                if len(ordered) > BLOCK:
                    corridor_candidates = ordered[np.linspace(0, len(ordered) - 1, BLOCK, dtype=np.int64)]
                else:
                    corridor_candidates = ordered
            else:
                goal_along = goal_lateral = 0.0
                goal_candidates = start_candidates
                corridor_candidates = np.arange(n, dtype=np.int64)

            uniform_candidates = np.unique(np.linspace(0, n - 1, min(n, BLOCK * 4), dtype=np.int64))
            used: set[int] = set()
            blocks = [
                padded(vectors, take_unique(start_candidates, used)),
                padded(vectors, take_unique(goal_candidates, used)),
                padded(vectors, take_unique(corridor_candidates, used)),
                padded(vectors, take_unique(uniform_candidates, used)),
            ]
            scene_tokens = np.concatenate(blocks, axis=0)
            tokens_all.append(scene_tokens)
            graphs.append(build_neighbor_graph(scene_tokens))
            contexts.append(np.asarray([
                goal_along / 100, goal_lateral / 100,
                math.hypot(goal_along, goal_lateral) / 100, float(has_goal),
                n / 4000, float(np.mean(types == 16)),
            ], dtype=np.float32))
            frames.append(np.asarray((x0, y0, ux, uy), dtype=np.float32))

            if is_train:
                target_x, target_y = parse_array(row[6]), parse_array(row[7])
                if len(target_x) != TRAJECTORY_STEPS or len(target_y) != TRAJECTORY_STEPS:
                    raise ValueError(f'Ожидалось 46 target-точек в строке {row_index}')
                local_target_x = (target_x - x0) * ux + (target_y - y0) * uy
                local_target_y = -(target_x - x0) * uy + (target_y - y0) * ux
                targets.append(np.stack((local_target_x, local_target_y), axis=1))
            if (row_index + 1) % 1000 == 0:
                print(f'extracted {row_index + 1} rows in {time.time() - started:.1f}s')

    payload = {
        'tokens': np.stack(tokens_all), 'context': np.stack(contexts),
        'frames': np.stack(frames), 'neighbors': np.stack(graphs),
    }
    if targets:
        payload['targets'] = np.stack(targets)
    np.savez(output_path, **payload)
    print('saved', output_path, payload['tokens'].shape)

if not TRAIN_CACHE.exists():
    extract_dataset(TRAIN_CSV, TRAIN_CACHE)
if not TEST_CACHE.exists():
    extract_dataset(TEST_CSV, TEST_CACHE)


## 3. Spatial split и метрика

Все сцены одной ячейки 100 × 100 метров остаются в одной части split. Это предотвращает утечку близких фрагментов карты.

In [ ]:
def spatial_split(frames: np.ndarray, cell_size: float = 100.0) -> tuple[np.ndarray, np.ndarray]:
    cells = np.floor(frames[:, :2] / cell_size).astype(np.int64)
    hashes = (cells[:, 0] * 73856093) ^ (cells[:, 1] * 19349663)
    validation = np.flatnonzero(np.mod(hashes, 5) == 0)
    training = np.flatnonzero(np.mod(hashes, 5) != 0)
    return training, validation

def ade_np(prediction: np.ndarray, truth: np.ndarray) -> float:
    return float(np.linalg.norm(prediction - truth, axis=-1).mean())

train_data = np.load(TRAIN_CACHE)
train_indices, validation_indices = spatial_split(train_data['frames'])
print('train:', len(train_indices), 'validation:', len(validation_indices))


## 4. Архитектура Graph Attention

Small и Big используют одинаковую структуру, но отличаются шириной, количеством graph-слоёв, latent-токенов и dropout.

In [ ]:
class AttentionBlock(nn.Module):
    def __init__(self, width: int, heads: int, dropout: float):
        super().__init__()
        self.cross_norm_q = nn.LayerNorm(width)
        self.cross_norm_kv = nn.LayerNorm(width)
        self.cross = nn.MultiheadAttention(width, heads, dropout=dropout, batch_first=True)
        self.self_norm = nn.LayerNorm(width)
        self.self_attention = nn.MultiheadAttention(width, heads, dropout=dropout, batch_first=True)
        self.ff_norm = nn.LayerNorm(width)
        self.ff = nn.Sequential(
            nn.Linear(width, width * 3), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(width * 3, width), nn.Dropout(dropout),
        )

    def forward(self, latent, tokens, padding_mask):
        query = self.cross_norm_q(latent)
        key_value = self.cross_norm_kv(tokens)
        latent = latent + self.cross(query, key_value, key_value, key_padding_mask=padding_mask, need_weights=False)[0]
        normalized = self.self_norm(latent)
        latent = latent + self.self_attention(normalized, normalized, normalized, need_weights=False)[0]
        return latent + self.ff(self.ff_norm(latent))

class VectorMapModel(nn.Module):
    def __init__(self, width, heads, latent_count, dropout, source_count=4, block_size=128):
        super().__init__()
        self.register_buffer('frequencies', torch.tensor([1.0, 2.0, 4.0, 8.0]))
        token_input_width = 5 + 2 * 2 * 4
        self.token_encoder = nn.Sequential(nn.Linear(token_input_width, width), nn.GELU(), nn.Linear(width, width))
        self.source_embedding = nn.Embedding(source_count, width)
        self.order_embedding = nn.Embedding(block_size, width)
        self.context_encoder = nn.Sequential(nn.Linear(6, width), nn.GELU(), nn.Linear(width, width))
        self.latents = nn.Parameter(torch.randn(latent_count, width) * 0.02)
        self.blocks = nn.ModuleList(AttentionBlock(width, heads, dropout) for _ in range(3))
        self.time_embedding = nn.Parameter(torch.randn(TRAJECTORY_STEPS, width) * 0.02)
        self.decode_norm_q = nn.LayerNorm(width)
        self.decode_norm_kv = nn.LayerNorm(width)
        self.decode_attention = nn.MultiheadAttention(width, heads, dropout=dropout, batch_first=True)
        decoder_layer = nn.TransformerEncoderLayer(
            d_model=width, nhead=heads, dim_feedforward=width * 3, dropout=dropout,
            activation='gelu', batch_first=True, norm_first=True,
        )
        self.temporal = nn.TransformerEncoder(decoder_layer, num_layers=2)
        self.output = nn.Sequential(nn.LayerNorm(width), nn.Linear(width, 2))
        self.register_buffer('source_ids', torch.arange(source_count).repeat_interleave(block_size))
        self.register_buffer('order_ids', torch.arange(block_size).repeat(source_count))

    def encode_tokens(self, tokens):
        coordinates = tokens[..., :2]
        angles = coordinates[..., None] * self.frequencies
        fourier = torch.cat((torch.sin(math.pi * angles), torch.cos(math.pi * angles)), dim=-1)
        expanded = torch.cat((tokens, fourier.flatten(-2)), dim=-1)
        encoded = self.token_encoder(expanded)
        return encoded + self.source_embedding(self.source_ids)[None] + self.order_embedding(self.order_ids)[None]

class GraphLayer(nn.Module):
    def __init__(self, width: int, heads: int, dropout: float):
        super().__init__()
        self.heads = heads
        self.head_width = width // heads
        self.norm = nn.LayerNorm(width)
        self.q, self.k, self.v = nn.Linear(width, width), nn.Linear(width, width), nn.Linear(width, width)
        self.edge_bias = nn.Sequential(nn.Linear(5, 48), nn.GELU(), nn.Linear(48, heads))
        self.output = nn.Sequential(nn.Linear(width, width), nn.Dropout(dropout))
        self.ff_norm = nn.LayerNorm(width)
        self.ff = nn.Sequential(nn.Linear(width, width * 3), nn.GELU(), nn.Dropout(dropout), nn.Linear(width * 3, width), nn.Dropout(dropout))

    def forward(self, values, raw_tokens, neighbors, padding_mask):
        batch, count, width = values.shape
        normalized = self.norm(values)
        batch_index = torch.arange(batch, device=values.device)[:, None, None]
        neighbor_values = normalized[batch_index, neighbors]
        neighbor_raw = raw_tokens[batch_index, neighbors]
        q = self.q(normalized).reshape(batch, count, self.heads, self.head_width)
        k = self.k(neighbor_values).reshape(batch, count, neighbors.shape[-1], self.heads, self.head_width)
        v = self.v(neighbor_values).reshape(batch, count, neighbors.shape[-1], self.heads, self.head_width)
        bias = self.edge_bias(neighbor_raw - raw_tokens[:, :, None, :])
        logits = (q[:, :, None] * k).sum(dim=-1) / math.sqrt(self.head_width) + bias
        neighbor_padding = padding_mask[batch_index, neighbors]
        logits = logits.masked_fill(neighbor_padding[..., None], -1e4)
        weights = torch.softmax(logits, dim=2)
        message = (weights[..., None] * v).sum(dim=2).reshape(batch, count, width)
        values = values + self.output(message)
        values = values + self.ff(self.ff_norm(values))
        return values.masked_fill(padding_mask[..., None], 0)

class GraphVectorModel(VectorMapModel):
    def __init__(self, width, heads, latent_count, dropout, graph_depth, graph_heads):
        super().__init__(width, heads, latent_count, dropout)
        self.graph_layers = nn.ModuleList(GraphLayer(width, graph_heads, dropout) for _ in range(graph_depth))

    def forward(self, tokens, context, neighbors):
        padding_mask = tokens.abs().sum(dim=-1) == 0
        token_values = self.encode_tokens(tokens)
        for layer in self.graph_layers:
            token_values = layer(token_values, tokens, neighbors, padding_mask)
        context_value = self.context_encoder(context)
        latent = self.latents[None].expand(len(tokens), -1, -1) + context_value[:, None]
        for block in self.blocks:
            latent = block(latent, token_values, padding_mask)
        time_values = self.time_embedding[None].expand(len(tokens), -1, -1)
        time_values = time_values + context_value[:, None] + latent.mean(dim=1)[:, None]
        memory = torch.cat((latent, token_values), dim=1)
        memory_mask = torch.cat((torch.zeros((len(tokens), latent.shape[1]), dtype=torch.bool, device=tokens.device), padding_mask), dim=1)
        query, key_value = self.decode_norm_q(time_values), self.decode_norm_kv(memory)
        time_values = time_values + self.decode_attention(query, key_value, key_value, key_padding_mask=memory_mask, need_weights=False)[0]
        return self.output(self.temporal(time_values))

def make_small_model() -> GraphVectorModel:
    return GraphVectorModel(width=192, heads=8, latent_count=28, dropout=0.06, graph_depth=4, graph_heads=6)

def make_big_model() -> GraphVectorModel:
    return GraphVectorModel(width=384, heads=8, latent_count=48, dropout=0.10, graph_depth=6, graph_heads=8)

print('small parameters:', sum(p.numel() for p in make_small_model().parameters()))
print('big parameters:', sum(p.numel() for p in make_big_model().parameters()))


## 5. Обучение и оценка

ADE вычисляется в метрах после обратного масштабирования target. Функция поддерживает map masking, velocity/curvature regularization, сохранение лучшей spatial-модели и полный resume состояния.

In [ ]:
def target_statistics(truth: np.ndarray, indices: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    mean = truth[indices].mean(axis=0, keepdims=True).astype(np.float32)
    std = truth[indices].std(axis=0, keepdims=True).astype(np.float32)
    std[std < 1] = 1
    return mean, std

@torch.inference_mode()
def predict_local(model, tokens, context, neighbors, mean, std, batch_size):
    model.eval()
    pieces = []
    mean_t, std_t = torch.from_numpy(mean).to(DEVICE), torch.from_numpy(std).to(DEVICE)
    for start in range(0, len(tokens), batch_size):
        batch_tokens = torch.from_numpy(tokens[start:start + batch_size]).to(DEVICE)
        batch_context = torch.from_numpy(context[start:start + batch_size]).to(DEVICE)
        batch_neighbors = torch.from_numpy(neighbors[start:start + batch_size].astype(np.int64)).to(DEVICE)
        pieces.append((model(batch_tokens, batch_context, batch_neighbors) * std_t + mean_t).cpu().numpy())
    return np.concatenate(pieces)

def make_loader(data, indices, mean, std, batch_size):
    target = (data['targets'].astype(np.float32) - mean) / std
    return DataLoader(TensorDataset(
        torch.from_numpy(data['tokens'][indices].astype(np.float32)),
        torch.from_numpy(data['context'][indices].astype(np.float32)),
        torch.from_numpy(data['neighbors'][indices].astype(np.int64)),
        torch.from_numpy(target[indices]),
    ), batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True)

def save_resume(path, epoch, model, optimizer, scheduler, scaler, extra=None):
    payload = {
        'epoch': epoch, 'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(), 'scaler': scaler.state_dict(),
        'python_rng': random.getstate(), 'numpy_rng': np.random.get_state(),
        'torch_rng': torch.get_rng_state(), 'cuda_rng': torch.cuda.get_rng_state_all(),
    }
    if extra:
        payload.update(extra)
    torch.save(payload, path)

def restore_resume(path, model, optimizer, scheduler, scaler):
    saved = torch.load(path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(saved['model'])
    optimizer.load_state_dict(saved['optimizer'])
    scheduler.load_state_dict(saved['scheduler'])
    scaler.load_state_dict(saved['scaler'])
    random.setstate(saved['python_rng'])
    np.random.set_state(saved['numpy_rng'])
    torch.set_rng_state(saved['torch_rng'].cpu())
    torch.cuda.set_rng_state_all(saved['cuda_rng'])
    return saved

def fit_spatial(model, data, train_idx, val_idx, mean, std, *, name, epochs, batch_size, lr, weight_decay, eta_min, mask_probability, curvature_weight, seed, eval_batch, include_initial=False):
    seed_all(seed)
    model = model.to(DEVICE)
    loader = make_loader(data, train_idx, mean, std, batch_size)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=eta_min)
    scaler = torch.amp.GradScaler('cuda')
    std_t = torch.from_numpy(std).to(DEVICE)
    best_path = ARTIFACT_DIR / f'{name}_best.pt'
    resume_path = ARTIFACT_DIR / f'{name}_resume.pt'
    prediction_path = ARTIFACT_DIR / f'{name}_validation.npz'
    best_score, best_epoch, start_epoch = float('inf'), -1, 1

    if include_initial:
        initial = predict_local(model, data['tokens'][val_idx].astype(np.float32), data['context'][val_idx].astype(np.float32), data['neighbors'][val_idx], mean, std, eval_batch)
        best_score, best_epoch = ade_np(initial, data['targets'][val_idx]), 0
        torch.save({'model': copy.deepcopy(model.state_dict()), 'mean': mean, 'std': std, 'best_score': best_score, 'best_epoch': 0}, best_path)
        np.savez(prediction_path, indices=val_idx, prediction=initial)

    if REUSE_EXISTING_ARTIFACTS and resume_path.exists():
        saved = restore_resume(resume_path, model, optimizer, scheduler, scaler)
        start_epoch = int(saved['epoch']) + 1
        best_score, best_epoch = float(saved['best_score']), int(saved['best_epoch'])

    began = time.time()
    for epoch in range(start_epoch, epochs + 1):
        model.train()
        total = 0.0
        for tokens, context, neighbors, target in loader:
            tokens, context = tokens.to(DEVICE, non_blocking=True), context.to(DEVICE, non_blocking=True)
            neighbors, target = neighbors.to(DEVICE, non_blocking=True), target.to(DEVICE, non_blocking=True)
            if mask_probability > 0:
                mask = torch.rand(tokens.shape[:2], device=DEVICE) < mask_probability
                mask[:, :8] = False
                tokens = tokens.masked_fill(mask[..., None], 0)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                output = model(tokens, context, neighbors)
                difference = (output - target) * std_t
                ade_loss = torch.sqrt(difference.square().sum(dim=-1) + 1e-3).mean()
                pred_velocity, true_velocity = output[:, 1:] - output[:, :-1], target[:, 1:] - target[:, :-1]
                loss = ade_loss + 0.08 * (pred_velocity - true_velocity).square().mean()
                if curvature_weight:
                    pred_curve = pred_velocity[:, 1:] - pred_velocity[:, :-1]
                    true_curve = true_velocity[:, 1:] - true_velocity[:, :-1]
                    loss = loss + curvature_weight * (pred_curve - true_curve).square().mean()
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5)
            scaler.step(optimizer)
            scaler.update()
            total += float(ade_loss.detach()) * len(tokens)
        scheduler.step()

        if epoch == 1 or epoch % 5 == 0 or epoch == epochs:
            prediction = predict_local(model, data['tokens'][val_idx].astype(np.float32), data['context'][val_idx].astype(np.float32), data['neighbors'][val_idx], mean, std, eval_batch)
            score = ade_np(prediction, data['targets'][val_idx])
            if score < best_score:
                best_score, best_epoch = score, epoch
                torch.save({'model': copy.deepcopy(model.state_dict()), 'mean': mean, 'std': std, 'best_score': score, 'best_epoch': epoch}, best_path)
                np.savez(prediction_path, indices=val_idx, prediction=prediction)
            save_resume(resume_path, epoch, model, optimizer, scheduler, scaler, {'best_score': best_score, 'best_epoch': best_epoch})
            print(f'{name} epoch={epoch} train={total / len(train_idx):.4f} val={score:.6f} best={best_score:.6f}@{best_epoch} seconds={time.time() - began:.1f}')

    best = torch.load(best_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(best['model'])
    return model, best


## 6. Spatial validation моделей

Эта стадия выбирает архитектуру и число эпох. Для быстрой генерации submission при уже известных настройках её можно отключить через `RUN_SPATIAL_VALIDATION = False` и использовать заранее полученные checkpoints.

In [ ]:
truth = train_data['targets'].astype(np.float32)
spatial_mean, spatial_std = target_statistics(truth, train_indices)

if RUN_SPATIAL_VALIDATION:
    seed_all(7703)
    small_model, small_result = fit_spatial(
        make_small_model(), train_data, train_indices, validation_indices, spatial_mean, spatial_std,
        name='small_spatial', epochs=180, batch_size=36, lr=4e-4, weight_decay=3e-4,
        eta_min=1e-5, mask_probability=0.0, curvature_weight=0.0, seed=7703, eval_batch=96,
    )

    seed_all(17011)
    augmented_small = make_small_model().to(DEVICE)
    augmented_small.load_state_dict(small_result['model'])
    augmented_small, aug_result = fit_spatial(
        augmented_small, train_data, train_indices, validation_indices, spatial_mean, spatial_std,
        name='small_aug_spatial', epochs=60, batch_size=36, lr=5e-5, weight_decay=1e-3,
        eta_min=3e-6, mask_probability=0.08, curvature_weight=0.02, seed=17011, eval_batch=96, include_initial=True,
    )

    del small_model
    torch.cuda.empty_cache()
    seed_all(22027)
    big_model, big_result = fit_spatial(
        make_big_model(), train_data, train_indices, validation_indices, spatial_mean, spatial_std,
        name='big_spatial', epochs=220, batch_size=24, lr=3e-4, weight_decay=8e-4,
        eta_min=8e-6, mask_probability=0.06, curvature_weight=0.02, seed=22027, eval_batch=48,
    )
    print('small augmented ADE:', aug_result['best_score'])
    print('big ADE:', big_result['best_score'])
else:
    print('Spatial training skipped; ожидаются готовые artifacts/*_validation.npz')


## 7. Пятифолдовая проверка ансамбля

Веса подбираются только по out-of-sample spatial predictions. Дополнительный 5-fold слой проверяет устойчивость двух весов.

In [ ]:
def fit_convex_weights(predictions: np.ndarray, truth: np.ndarray) -> np.ndarray:
    count = len(predictions)
    def objective(weights):
        return ade_np(np.tensordot(weights, predictions, axes=(0, 0)), truth)
    result = minimize(
        objective, np.full(count, 1 / count), method='SLSQP',
        bounds=[(0, 1)] * count,
        constraints={'type': 'eq', 'fun': lambda weights: weights.sum() - 1},
        options={'maxiter': 500, 'ftol': 1e-11},
    )
    if not result.success:
        raise RuntimeError(result.message)
    return result.x

big_validation = np.load(ARTIFACT_DIR / 'big_spatial_validation.npz')
aug_validation = np.load(ARTIFACT_DIR / 'small_aug_spatial_validation.npz')
assert np.array_equal(big_validation['indices'], aug_validation['indices'])
blend_indices = big_validation['indices']
blend_predictions = np.stack((big_validation['prediction'], aug_validation['prediction'])).astype(np.float64)
blend_truth = truth[blend_indices].astype(np.float64)
fold_ids = np.mod(blend_indices * 2654435761, 5)
oof_prediction = np.empty_like(blend_truth)
fold_weights = []
for fold in range(5):
    fold_train, fold_test = fold_ids != fold, fold_ids == fold
    weights = fit_convex_weights(blend_predictions[:, fold_train], blend_truth[fold_train])
    fold_weights.append(weights)
    oof_prediction[fold_test] = np.tensordot(weights, blend_predictions[:, fold_test], axes=(0, 0))
    print(f'fold={fold}: big={weights[0]:.6f}, augmented_small={weights[1]:.6f}')

final_weights = fit_convex_weights(blend_predictions, blend_truth)
full_blend = np.tensordot(final_weights, blend_predictions, axes=(0, 0))
print('5-fold OOF ADE:', ade_np(oof_prediction, blend_truth))
print('full holdout ADE:', ade_np(full_blend, blend_truth))
print('final weights:', final_weights)
np.savez(ARTIFACT_DIR / 'blend_weights.npz', weights=final_weights, fold_weights=np.stack(fold_weights))


## 8. Переобучение на всех данных

После выбора числа эпох модели создаются заново. Validation-веса не переносятся. Small base обучается 180 эпох, затем проходит 55 эпох masking fine-tune. Big обучается 185 эпох — это лучший epoch spatial run.

In [ ]:
def train_full(model, data, mean, std, *, name, epochs, scheduler_epochs, batch_size, lr, weight_decay, eta_min, mask_probability, curvature_weight, seed):
    seed_all(seed)
    model = model.to(DEVICE)
    all_indices = np.arange(len(data['tokens']))
    loader = make_loader(data, all_indices, mean, std, batch_size)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=scheduler_epochs, eta_min=eta_min)
    scaler = torch.amp.GradScaler('cuda')
    std_t = torch.from_numpy(std).to(DEVICE)
    resume_path = ARTIFACT_DIR / f'{name}_resume.pt'
    final_path = ARTIFACT_DIR / f'{name}.pt'
    start_epoch = 1
    if REUSE_EXISTING_ARTIFACTS and final_path.exists():
        saved = torch.load(final_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(saved['model'])
        print('loaded completed model:', final_path)
        return model
    if REUSE_EXISTING_ARTIFACTS and resume_path.exists():
        saved = restore_resume(resume_path, model, optimizer, scheduler, scaler)
        start_epoch = int(saved['epoch']) + 1

    began = time.time()
    for epoch in range(start_epoch, epochs + 1):
        model.train()
        total = 0.0
        for tokens, context, neighbors, target in loader:
            tokens, context = tokens.to(DEVICE, non_blocking=True), context.to(DEVICE, non_blocking=True)
            neighbors, target = neighbors.to(DEVICE, non_blocking=True), target.to(DEVICE, non_blocking=True)
            if mask_probability > 0:
                mask = torch.rand(tokens.shape[:2], device=DEVICE) < mask_probability
                mask[:, :8] = False
                tokens = tokens.masked_fill(mask[..., None], 0)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                output = model(tokens, context, neighbors)
                difference = (output - target) * std_t
                ade_loss = torch.sqrt(difference.square().sum(dim=-1) + 1e-3).mean()
                pred_velocity, true_velocity = output[:, 1:] - output[:, :-1], target[:, 1:] - target[:, :-1]
                loss = ade_loss + 0.08 * (pred_velocity - true_velocity).square().mean()
                if curvature_weight:
                    pred_curve = pred_velocity[:, 1:] - pred_velocity[:, :-1]
                    true_curve = true_velocity[:, 1:] - true_velocity[:, :-1]
                    loss = loss + curvature_weight * (pred_curve - true_curve).square().mean()
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5)
            scaler.step(optimizer)
            scaler.update()
            total += float(ade_loss.detach()) * len(tokens)
        scheduler.step()
        if epoch == 1 or epoch % 5 == 0 or epoch == epochs:
            save_resume(resume_path, epoch, model, optimizer, scheduler, scaler)
            print(f'{name} epoch={epoch} train_ADE={total / len(all_indices):.4f} seconds={time.time() - began:.1f}')

    torch.save({'model': model.state_dict(), 'mean': mean, 'std': std, 'epochs': epochs}, final_path)
    return model

all_indices = np.arange(len(train_data['targets']))
full_mean, full_std = target_statistics(truth, all_indices)

if RUN_FULL_TRAINING:
    seed_all(7703)
    small_full = train_full(
        make_small_model(), train_data, full_mean, full_std, name='small_full',
        epochs=180, scheduler_epochs=180, batch_size=36, lr=4e-4, weight_decay=3e-4, eta_min=1e-5,
        mask_probability=0.0, curvature_weight=0.0, seed=7703,
    )
    seed_all(17011)
    augmented_small_full = make_small_model().to(DEVICE)
    augmented_small_full.load_state_dict(small_full.state_dict())
    augmented_small_full = train_full(
        augmented_small_full, train_data, full_mean, full_std, name='small_aug_full',
        epochs=55, scheduler_epochs=60, batch_size=36, lr=5e-5, weight_decay=1e-3, eta_min=3e-6,
        mask_probability=0.08, curvature_weight=0.02, seed=17011,
    )
    del small_full
    torch.cuda.empty_cache()
    seed_all(22027)
    big_full = train_full(
        make_big_model(), train_data, full_mean, full_std, name='big_full',
        epochs=185, scheduler_epochs=220, batch_size=24, lr=3e-4, weight_decay=8e-4, eta_min=8e-6,
        mask_probability=0.06, curvature_weight=0.02, seed=22027,
    )


## 9. Inference, обратное преобразование и submission

Ансамблирование выполняется в локальной системе координат. Затем 46 точек возвращаются в глобальные координаты.

In [ ]:
test_data = np.load(TEST_CACHE)
if not RUN_FULL_TRAINING:
    augmented_small_full = make_small_model().to(DEVICE)
    augmented_small_full.load_state_dict(torch.load(ARTIFACT_DIR / 'small_aug_full.pt', map_location=DEVICE, weights_only=False)['model'])
    big_full = make_big_model().to(DEVICE)
    big_full.load_state_dict(torch.load(ARTIFACT_DIR / 'big_full.pt', map_location=DEVICE, weights_only=False)['model'])

test_tokens = test_data['tokens'].astype(np.float32)
test_context = test_data['context'].astype(np.float32)
test_neighbors = test_data['neighbors'].astype(np.int64)
big_local = predict_local(big_full, test_tokens, test_context, test_neighbors, full_mean, full_std, 40)
small_local = predict_local(augmented_small_full, test_tokens, test_context, test_neighbors, full_mean, full_std, 64)
local_prediction = final_weights[0] * big_local + final_weights[1] * small_local

global_prediction = np.empty_like(local_prediction)
for scene, (x0, y0, ux, uy) in enumerate(test_data['frames']):
    along, lateral = local_prediction[scene, :, 0], local_prediction[scene, :, 1]
    global_prediction[scene, :, 0] = x0 + along * ux - lateral * uy
    global_prediction[scene, :, 1] = y0 + along * uy + lateral * ux

def format_number(value: float) -> str:
    text = f'{float(value):.8f}'.rstrip('0').rstrip('.')
    return '0' if text in ('-0', '') else text

def format_array(values: np.ndarray) -> str:
    return '[' + ' '.join(format_number(value) for value in values) + ']'

submission_path = ARTIFACT_DIR / 'submission.csv'
with submission_path.open('w', encoding='utf-8', newline='') as stream:
    writer = csv.writer(stream, lineterminator='\n')
    writer.writerow(('pred_x', 'pred_y'))
    for trajectory in global_prediction:
        writer.writerow((format_array(trajectory[:, 0]), format_array(trajectory[:, 1])))

assert len(global_prediction) == len(test_data['tokens'])
assert global_prediction.shape[1:] == (46, 2)
assert np.isfinite(global_prediction).all()
print('submission:', submission_path.resolve())
print('rows:', len(global_prediction), 'shape:', global_prediction.shape, 'weights:', final_weights)


## Итог

Файл `artifacts/submission.csv` готов к отправке. Перед публикацией репозитория не добавляйте исходные данные, NPZ-кэши и checkpoints: они исключены через `.gitignore`.